# Musanga — Optimize and upload UAV RGB and probability maps to Earth Engine

**Colab / source Drive account:** `kristof.vanoost@uclouvain.be` · **Earth Engine project:** `musanga-511110`

## Workflow
1. Authenticate and mount your university Google Drive.
2. Inspect the source TIFFs; choose Baego first.
3. **Optimize before upload:** lossless, tiled, DEFLATE-compressed native-resolution RGB (first 3 bands); convert probability `uint16` (0–10000, 65535 NoData) to `uint8` (0–200, 255 NoData), with 0.5-percentage-point quantization, and validate sample windows. **No separate binary raster**: binary is computed in the viewer.
4. Use the private Cloud Storage bucket created *once* by the Musanga project owner (`vanoost@gmail.com`) and shared with your university account via **Storage Object Admin** on the bucket.
5. Stage optimized TIFFs in GCS and submit Earth Engine ingestion tasks. Source TIFFs are never changed.

**Permissions:** Your EE access already works. `storage.buckets.create` is **not required** if the owner creates the staging bucket first. This notebook does not create GCS buckets and will show a clear permission error if the bucket is unavailable. EE asset-writing permission is still needed.

**Outputs:** optimized TIFFs are temporarily stored on Colab local disk (`/content/musanga_gee_optimized`) to avoid copying large derivatives back to Google Drive. Uploads are asynchronous and can incur storage/network charges. If Colab disconnects before GCS staging completes, local optimized TIFFs may need rebuilding.

**Viewer:** [Musanga UAV probability viewer](https://github.com/kvanoost/Musanga/blob/main/gee/Musanga_UAV_probability_viewer.js). It expects the `uint8` probability encoding produced by this version. Previously ingested `uint16` assets are not interchangeable with the updated viewer.


In [ ]:
# Keep Colab's pinned authentication library; avoid '-U' upgrading google-auth.
%pip -q install "google-auth==2.49.0" earthengine-api google-cloud-storage rasterio
# If you already installed incompatible versions in this session, restart the runtime once.


In [ ]:
from pathlib import Path
import google.auth
from google.colab import auth, drive
import ee, rasterio
from google.cloud import storage

CLOUD_PROJECT = "musanga-511110"
ACCOUNT = "kristof.vanoost@uclouvain.be"  # Use Colab Pro / UCLouvain credentials
BUCKET_NAME = "musanga-511110-musanga-uav-stage"  # Owner creates bucket once; UCLouvain account uploads objects
ASSET_FOLDER = f"projects/{CLOUD_PROJECT}/assets/musanga_uav"

auth.authenticate_user()
creds, _ = google.auth.default(scopes=[
    "https://www.googleapis.com/auth/cloud-platform",
    "https://www.googleapis.com/auth/earthengine"
])
ee.Initialize(credentials=creds, project=CLOUD_PROJECT)
client = storage.Client(project=CLOUD_PROJECT, credentials=creds)
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
GEO = ROOT/"data"/"Geo_Ref"
MAPS = ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"
print("Earth Engine project:", CLOUD_PROJECT)
print("Expected Google account:", ACCOUNT)
print("Asset folder:", ASSET_FOLDER)

In [ ]:
# Enable Baego first; add sites when ready.
SELECTED_SITES = ["Baego"]
# SELECTED_SITES = ["Baego", "Yoko", "Yangambi", "MOS_1_2"]

SITE_RASTERS = {
    "Baego": "Baego_georef.tif",
    "Yoko": "Yoko_kvo_georef.tif",
    "Yangambi": "Yangambi_georef.tif",
    "MOS_1_2": "MOS_1_2_kvo.tif",
}

def inspect_site(site):
    rgb=GEO/SITE_RASTERS[site]
    prob=MAPS/f"{site}_musanga_probability_uint16.tif"
    for label,path in [("RGB",rgb),("probability",prob)]:
        if not path.exists():
            raise FileNotFoundError(f"{site} {label} missing: {path}")
        with rasterio.open(path) as src:
            print(site,label, "bands",src.count,"dtype",src.dtypes,"CRS",src.crs,
                  "resolution",src.res,"NoData",src.nodata,
                  "size GB",round(path.stat().st_size/1e9,2))
            assert src.crs is not None, f"Missing CRS: {path}"
            if label=="RGB":
                assert src.count>=3, f"RGB needs at least 3 bands: {path}"
            else:
                assert src.count==1 and src.dtypes[0]=="uint16", f"Expected one uint16 probability band: {path}"
    return rgb,prob

SOURCES={site:inspect_site(site) for site in SELECTED_SITES}
print("Inputs verified. Next: run the optimization cell BEFORE staging.")

## Block 4 — Compression and probability quantization

Run this **before** creating/staging the bucket and ingesting assets. RGB remains native resolution, lossless. Probability is uint8 with a 0–200 scale (0.5 percentage-point steps), and NoData 255. The original uint16 probability TIFF remains unchanged on Drive.


In [ ]:
# BLOCK 4 — Optimize all selected rasters BEFORE GCS upload (windowed, low memory)
# Keeps RGB at native pixel size, first 3 bands, losslessly compressed.
# Converts probability 0..10000 uint16 -> 0..200 uint8; 255=NoData; binary stays virtual.
from pathlib import Path
import numpy as np, rasterio
from rasterio.windows import Window
from rasterio.enums import ColorInterp

OPT_DIR=Path("/content/musanga_gee_optimized")
OPT_DIR.mkdir(parents=True,exist_ok=True)
OPTIMIZED={}
P_SCALE=200   # 0.5 percentage-point bins, threshold in GEE uses /200
P_NODATA=255

def windows(width,height,block=512):
    for row in range(0,height,block):
        for col in range(0,width,block):
            yield Window(col,row,min(block,width-col),min(block,height-row))

def optimize_rgb(inp,out):
    with rasterio.open(inp) as src:
        assert src.count>=3, f"RGB needs >=3 bands: {inp}"
        assert src.dtypes[0]=="uint8", (
            f"RGB is {src.dtypes[0]}; refusing automatic 16-bit stretch. "
            "Choose visualization stretch explicitly before uint8 conversion.")
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=3,dtype="uint8",nodata=None,
                       compress="DEFLATE",predictor=2,tiled=True,
                       blockxsize=512,blockysize=512,interleave="pixel",BIGTIFF="IF_SAFER")
        with rasterio.open(out,"w",**profile) as dst:
            dst.colorinterp=(ColorInterp.red,ColorInterp.green,ColorInterp.blue)
            has_alpha=(src.count>=4 and src.colorinterp[3]==ColorInterp.alpha)
            for w in windows(src.width,src.height):
                rgb=src.read([1,2,3],window=w)
                # Preserve band-level validity and alpha transparency while dropping alpha band.
                validity=np.logical_and.reduce(src.read_masks([1,2,3],window=w)>0,axis=0)
                if has_alpha: validity &= src.read(4,window=w)>0
                dst.write(rgb,window=w)
                dst.write_mask((validity*255).astype("uint8"),window=w)
    return out

def optimize_probability(inp,out):
    with rasterio.open(inp) as src:
        assert src.count==1 and src.dtypes[0]=="uint16"
        profile=src.profile.copy()
        profile.update(driver="GTiff",count=1,dtype="uint8",nodata=P_NODATA,
                       compress="DEFLATE",predictor=2,tiled=True,
                       blockxsize=512,blockysize=512,BIGTIFF="IF_SAFER")
        with rasterio.open(out,"w",**profile) as dst:
            dst.set_band_description(1,"probability_uint8_scale_200")
            for w in windows(src.width,src.height):
                values=src.read(1,window=w)
                valid=(src.read_masks(1,window=w)>0)&(values!=65535)
                if np.any(valid):
                    assert np.max(values[valid])<=10000, "Unexpected probability >10000"
                scaled=np.full(values.shape,P_NODATA,dtype="uint8")
                scaled[valid]=np.rint(values[valid].astype("float32")*(P_SCALE/10000)).clip(0,P_SCALE).astype("uint8")
                dst.write(scaled,1,window=w)
    return out

def quick_validate(source,target,kind):
    with rasterio.open(source) as a, rasterio.open(target) as b:
        assert a.width==b.width and a.height==b.height and a.crs==b.crs
        assert a.transform.almost_equals(b.transform), "GeoTransform differs"
        if kind=="RGB":
            assert b.count==3 and b.dtypes[0]=="uint8"
        else:
            assert b.count==1 and b.dtypes[0]=="uint8" and b.nodata==255
        # Sample spatially separated windows, including corners.
        coords=[(0,0),(max(0,a.width//2-128),max(0,a.height//2-128)),
                (max(0,a.width-256),max(0,a.height-256))]
        for col,row in coords:
            w=Window(col,row,min(256,a.width-col),min(256,a.height-row))
            if kind=="probability":
                inp=a.read(1,window=w)
                out=b.read(1,window=w)
                ok=(a.read_masks(1,window=w)>0)&(inp!=65535)
                assert np.all(out[~ok]==255), "NoData mismatch"
                if ok.any():
                    err=np.abs(out[ok].astype("float32")/P_SCALE-inp[ok].astype("float32")/10000)
                    assert err.max()<=0.002501, f"Quantization error {err.max()}"
            else:
                assert np.array_equal(a.read([1,2,3],window=w),b.read([1,2,3],window=w))
    print(kind, "validation PASS",f"{source.stat().st_size/1e9:.2f} -> {target.stat().st_size/1e9:.2f} GB",
          f"({target.stat().st_size/source.stat().st_size:.0%} of original)")

for site,(rgb,prob) in SOURCES.items():
    rgb_out=OPT_DIR/f"{site}_RGB_optimized.tif"
    prob_out=OPT_DIR/f"{site}_Musanga_probability_uint8_optimized.tif"
    # Reuse derivatives when present; validation is still repeated.
    if not rgb_out.exists(): optimize_rgb(rgb,rgb_out)
    quick_validate(rgb,rgb_out,"RGB")
    if not prob_out.exists(): optimize_probability(prob,prob_out)
    quick_validate(prob,prob_out,"probability")
    OPTIMIZED[site]={"RGB":rgb_out,"Musanga_probability":prob_out}
print("READY:",OPTIMIZED)
print("Original source rasters on Drive were not modified.")


In [ ]:
# BLOCK 5 — Reuse owner-created private GCS bucket and ensure EE asset folder
# The project owner vanoost@gmail.com must create this bucket ONCE through Cloud Console.
# Grant kristof.vanoost@uclouvain.be Storage Object Admin on the bucket.
from google.api_core.exceptions import Forbidden, NotFound
import json
try:
    bucket=client.get_bucket(BUCKET_NAME)
    print("GCS bucket available:",bucket.name)
except (Forbidden,NotFound) as exc:
    raise RuntimeError(
        f"GCS bucket '{BUCKET_NAME}' is not accessible. Sign in to Cloud Console as "
        "vanoost@gmail.com, create it in project musanga-511110, and grant "
        "kristof.vanoost@uclouvain.be Storage Object Admin on THIS bucket. "
        "Then rerun this cell. No project-wide Storage Admin role is necessary."
    ) from exc
try:
    ee.data.getAsset(ASSET_FOLDER)
    print("Existing Earth Engine folder:",ASSET_FOLDER)
except ee.EEException as err:
    if "not found" not in str(err).lower() and "does not exist" not in str(err).lower():
        raise
    ee.data.createAsset({"type":"FOLDER"},ASSET_FOLDER)
    print("Created Earth Engine folder:",ASSET_FOLDER)


In [ ]:
# BLOCK 6 — Stage optimized TIFFs and ingest assets
# Validate first and run ONLY after Block 4 and Block 5.
# Existing same-size GCS objects and completed EE assets are skipped.
import json
START_INGESTION=True
assert OPTIMIZED, "Run optimization (Block 4) first."

def asset_exists(asset_id):
    try:
        ee.data.getAsset(asset_id)
        return True
    except ee.EEException as exc:
        if "not found" in str(exc).lower() or "does not exist" in str(exc).lower():
            return False
        raise

def stage_file(local_path,object_name):
    blob=bucket.blob(object_name)
    size=local_path.stat().st_size
    if blob.exists(client=client):
        blob.reload(client=client)
        if blob.size==size:
            print("GCS already staged:",object_name)
            return f"gs://{bucket.name}/{object_name}"
        raise ValueError(f"Existing GCS object has different byte size: {object_name}")
    print(f"Uploading optimized {local_path.name} ({size/1e9:.2f} GB)")
    blob.chunk_size=8*1024*1024
    blob.upload_from_filename(str(local_path),timeout=3600,checksum="auto")
    return f"gs://{bucket.name}/{object_name}"

submitted=[]
for site,files in OPTIMIZED.items():
    for kind,path in files.items():
        asset=f"{ASSET_FOLDER}/{site}_{kind}"
        if asset_exists(asset):
            print("Existing EE asset, skipping:",asset)
            continue
        # Namespace differentiates this optimized encoding from any previous staging.
        uri=stage_file(path,f"musanga_uav/v2_uint8/{site}/{path.name}")
        manifest={"name":asset,
                  "tilesets":[{"id":"t0","sources":[{"uris":[uri]}]}],
                  "properties":{"site":site,"dataset":"Musanga UAV",
                                "source_file":path.name,"upload_version":"optimized_v2"}}
        if kind=="RGB":
            manifest["bands"]=[
                {"id":label,"tilesetId":"t0","tilesetBandIndex":i,"pyramidingPolicy":"MEAN"}
                for i,label in enumerate(["red","green","blue"])
            ]
        else:
            manifest["bands"]=[{"id":"probability_uint8","tilesetId":"t0",
                "tilesetBandIndex":0,"missingData":{"values":[255]},
                "pyramidingPolicy":"MEAN"}]
            manifest["properties"]["probability_scale"]=200
            manifest["properties"]["probability_nodata"]=255
        if START_INGESTION:
            result=ee.data.startIngestion(None,manifest)
            submitted.append(dict(site=site,kind=kind,asset=asset,result=result))
            print("Submitted Earth Engine task:",asset,result)
        else:
            print("DRY RUN:",json.dumps(manifest,indent=2))
print("Ingestions submitted:",len(submitted))
print("EE image assets become available once ingestion finishes.")


In [ ]:
# BLOCK 7 — Run this cell again later to check progress. The uploads are run by Earth Engine.
import pandas as pd
task_rows=[]
for t in ee.data.getTaskList():
    desc=t.get("description","")
    task_id=t.get("id","")
    # Ingestion descriptions may omit the asset ID; show all ingestion tasks.
    if t.get("task_type")=="INGEST" or t.get("type")=="INGEST" or "Musanga" in desc:
        task_rows.append({k:t.get(k) for k in ("id","state","description","error_message")})
display(pd.DataFrame(task_rows))
for site in SELECTED_SITES:
    for suffix in ("RGB","Musanga_probability"):
        asset=f"{ASSET_FOLDER}/{site}_{suffix}"
        print(site,suffix,"READY" if asset_exists(asset) else "not yet available")
print("Once both site assets are READY, open the viewer script in the GEE Code Editor.")

## Viewer

Open the source [Musanga UAV probability viewer](https://github.com/kvanoost/Musanga/blob/main/gee/Musanga_UAV_probability_viewer.js), copy it to the [GEE Code Editor](https://code.earthengine.google.com/) and press **Run**. The control panel includes site selection, binary threshold, transparency, and layer toggles.

To share with colleagues: grant them access to the Earth Engine assets (or share via an Earth Engine App) and make sure they can access the Cloud project. Merely sending the JavaScript link does not grant access to private assets.
